# 05 · Model Interpretation and Business Use

This notebook compares the two decision points and translates model scores into a simple prioritization view.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

METRIC_DIR = Path('../outputs/model_metrics')
INTERP_DIR = Path('../outputs/interpretation')
PRED_DIR = Path('../outputs/predictions')

comparison = pd.read_csv(METRIC_DIR / 'placement_vs_delivery.csv')
importance = pd.read_csv(INTERP_DIR / 'catboost_feature_importance.csv')
comparison.round(3)

The placement model keeps more recall because it casts a wider net. The delivery model is more precise and ranks negative-review cases much better, which is reflected in both ROC-AUC and PR-AUC.

In [ ]:
importance.head(12)

The strongest delivery-stage features are mostly related to fulfillment timing and order complexity. These are predictive associations. They should not be read as causal estimates of what would happen if one variable were changed.

## Capture rate and lift

For an operational team, ranking quality is often easier to use than a single classification threshold. I therefore check how many negative reviews are found inside the highest-risk 5%, 10% and 20% of orders.

In [ ]:
def capture_table(stage, y, prob):
    y = np.asarray(y)
    prob = np.asarray(prob)
    base_rate = y.mean()
    order = np.argsort(-prob)
    y_sorted = y[order]

    rows = []
    for share in [0.05, 0.10, 0.20]:
        n = max(1, int(np.ceil(len(y) * share)))
        selected = y_sorted[:n]
        rows.append({
            'stage': stage,
            'top_share': share,
            'orders': n,
            'negative_reviews_captured': selected.sum() / y.sum(),
            'segment_negative_review_rate': selected.mean(),
            'lift_vs_average': selected.mean() / base_rate
        })
    return rows

placement = pd.read_csv(PRED_DIR / 'placement_predictions.csv')
delivery = pd.read_csv(PRED_DIR / 'delivery_predictions.csv')

lift = pd.DataFrame(
    capture_table(
        'At Placement',
        placement['actual_review_bad'],
        placement['placement_probability']
    )
    +
    capture_table(
        'At Delivery',
        delivery['actual_review_bad'],
        delivery['delivery_probability']
    )
)

lift.to_csv(METRIC_DIR / 'lift_capture.csv', index=False)
lift.round(3)

The delivery model is especially useful for prioritization. In the current test set, the top **10%** of delivery-risk scores contain about **43.6%** of all negative reviews, with a negative-review rate of about **55.9%**. That is roughly **4.4 times** the overall rate.

This does not establish the value of contacting those customers. It shows that the score concentrates risk well enough to support a targeted service-recovery experiment.

## Capacity and decision-cost analysis

A ranking model becomes more useful when it is tied to an operating constraint. I therefore evaluate how many negative reviews are captured when a customer-service team can only inspect a fixed share of orders. I also use a normalized error-cost model to show when earlier intervention could be worth accepting more false positives.


In [ ]:
delivery_pred = pd.read_csv('../outputs/predictions/delivery_predictions.csv')
y = delivery_pred['actual_review_bad'].astype(int).to_numpy()
p = delivery_pred['delivery_probability'].to_numpy()
n = len(delivery_pred)
total_bad = y.sum()
base_rate = y.mean()

capacity_rows = []
previous_capture = 0
for share in [0.01, 0.025, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40]:
    k = max(1, int(np.ceil(n * share)))
    idx = np.argsort(-p)[:k]
    capture = y[idx].sum() / total_bad
    queue_rate = y[idx].mean()
    capacity_rows.append({
        'capacity_share': share,
        'orders_reviewed': k,
        'negative_reviews_captured': capture,
        'queue_negative_review_rate': queue_rate,
        'lift_vs_average': queue_rate / base_rate,
        'marginal_capture_vs_prior_row': capture - previous_capture
    })
    previous_capture = capture

capacity_curve = pd.DataFrame(capacity_rows)
capacity_curve.to_csv('../outputs/model_metrics/capacity_curve.csv', index=False)
capacity_curve


In [ ]:
def error_counts(pred_df, prob_col, threshold_col):
    y_true = pred_df['actual_review_bad'].astype(int).to_numpy()
    threshold = float(pred_df[threshold_col].iloc[0])
    y_hat = (pred_df[prob_col].to_numpy() >= threshold).astype(int)
    fp = int(((y_hat == 1) & (y_true == 0)).sum())
    fn = int(((y_hat == 0) & (y_true == 1)).sum())
    tp = int(((y_hat == 1) & (y_true == 1)).sum())
    tn = int(((y_hat == 0) & (y_true == 0)).sum())
    return threshold, tp, fp, fn, tn

placement_pred = pd.read_csv('../outputs/predictions/placement_predictions.csv')
rows = []
stage_specs = [
    ('At Placement', placement_pred, 'placement_probability', 'placement_threshold'),
    ('At Delivery', delivery_pred, 'delivery_probability', 'delivery_threshold')
]
for stage, frame, prob_col, threshold_col in stage_specs:
    threshold, tp, fp, fn, tn = error_counts(frame, prob_col, threshold_col)
    for fn_cost in [1, 2, 5, 10, 20]:
        total_cost = fp + fn_cost * fn
        rows.append({
            'stage': stage,
            'false_positive_cost_units': 1,
            'false_negative_cost_units': fn_cost,
            'threshold': threshold,
            'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
            'normalized_total_cost': total_cost,
            'cost_units_per_1000_orders': total_cost / len(frame) * 1000
        })
cost_sensitivity = pd.DataFrame(rows)
cost_sensitivity.to_csv('../outputs/model_metrics/cost_sensitivity.csv', index=False)
cost_sensitivity


In [ ]:
_, _, fp_p, fn_p, _ = error_counts(placement_pred, 'placement_probability', 'placement_threshold')
_, _, fp_d, fn_d, _ = error_counts(delivery_pred, 'delivery_probability', 'delivery_threshold')
break_even_ratio = (fp_p - fp_d) / (fn_d - fn_p)
break_even = pd.DataFrame([{
    'placement_fp': fp_p,
    'placement_fn': fn_p,
    'delivery_fp': fp_d,
    'delivery_fn': fn_d,
    'break_even_fn_to_fp_cost_ratio': break_even_ratio
}])
break_even.to_csv('../outputs/model_metrics/cost_breakeven.csv', index=False)
break_even


The delivery score is highly concentrated at the top of the queue: reviewing the highest-risk **5%** of test orders captures about **29.8%** of all negative reviews, while the top **10%** captures about **43.6%**. The value of additional capacity falls as lower-risk orders enter the queue.

The cost analysis is deliberately expressed in normalized units rather than invented currency values. At the validation-selected thresholds, the placement model creates many more false positives but slightly fewer false negatives. Under this simplified two-error model, placement becomes cheaper only when one false negative is worth more than about **32.7 false positives**. This is not a claim about Olist's real economics; it shows exactly what cost assumption would be needed to justify the earlier intervention point.
